# Video RAG — Phase 1: Notebook 03
## Dense Embeddings & FAISS Vector Indexing

In this notebook, we:
1. Generate dense vector embeddings for each chunk using **`BAAI/bge-small-en-v1.5`**.
2. Apply **L2 Normalization** on embeddings.
3. Build a **FAISS `IndexFlatIP`** vector index (Inner Product = exact Cosine Similarity on normalized vectors).
4. Persist the vector index and associated metadata for downstream retrieval.


In [ ]:
import sys
from pathlib import Path
import numpy as np

sys.path.append(str(Path.cwd().parent))

from src.chunking import TranscriptChunker
from src.embeddings import BGEEmbedder
from src.vector_store import FAISSVectorStore


### Step 1: Load Chunks


In [ ]:
chunks = TranscriptChunker.load_chunks("../data/transcripts/sample_chunks.json")
print(f"Loaded {len(chunks)} chunks for indexing.")


### Step 2: Initialize BGE Embedding Model


In [ ]:
embedder = BGEEmbedder(model_name="BAAI/bge-small-en-v1.5", normalize_embeddings=True)
print(f"Embedding Model: {embedder.model_name}")
print(f"Vector Dimension: {embedder.dimension}")


### Step 3: Compute Embeddings for Chunks


In [ ]:
chunk_embeddings = embedder.embed_chunks(chunks, show_progress_bar=True)
print(f"Chunk Embeddings Matrix Shape: {chunk_embeddings.shape}")
print(f"L2 Norm Verification (first vector): {np.linalg.norm(chunk_embeddings[0]):.4f}")


### Step 4: Ingest into FAISS Vector Store


In [ ]:
vector_store = FAISSVectorStore(dimension=embedder.dimension)
vector_store.add_chunks(chunks, chunk_embeddings)
print(f"Total Vectors Indexed: {vector_store.total_vectors}")


### Step 5: Test Similarity Search on Sample Query


In [ ]:
test_query = "What is database normalization?"
query_vec = embedder.embed_query(test_query)
results = vector_store.search(query_vec, top_k=3)

print(f"Query: '{test_query}'\n")
for rank, res in enumerate(results, 1):
    print(f"Rank {rank} (Cosine Similarity Score: {res['score']:.4f}):")
    print(f"  Chunk ID: {res['chunk_id']} | Time: [{res['start_time']}s - {res['end_time']}s]")
    print(f"  Text: {res['text'][:120]}...\n")


### Step 6: Save FAISS Index and Metadata


In [ ]:
vector_store.save(
    index_path="../data/vector_store/faiss.index",
    metadata_path="../data/vector_store/chunks_metadata.json"
)
print("Saved FAISS index and metadata successfully.")
